# Python Syntax, Coding Conventions, and Style

A reference for writing Python that is correct, readable, and maintainable. It covers the language's syntax rules, the conventions the community has settled on (PEP 8 and friends), the best practices that follow from them, and the pitfalls that trip up beginners and experienced programmers alike.

Examples target **Python 3.10+** unless noted.

---

## Table of Contents

1. [How to Read This Document](#1-how-to-read-this-document)
2. [Lexical Structure](#2-lexical-structure)
3. [Names and Naming Conventions](#3-names-and-naming-conventions)
4. [Layout and Whitespace](#4-layout-and-whitespace)
5. [Expressions and Operators](#5-expressions-and-operators)
6. [Statements and Control Flow](#6-statements-and-control-flow)
7. [Functions](#7-functions)
8. [Data Structures and Idioms](#8-data-structures-and-idioms)
9. [Classes and Object Model](#9-classes-and-object-model)
10. [Modules, Packages, and Imports](#10-modules-packages-and-imports)
11. [Errors and Exceptions](#11-errors-and-exceptions)
12. [Comments, Docstrings, and Documentation](#12-comments-docstrings-and-documentation)
13. [Type Hints](#13-type-hints)
14. [Files, Resources, and Context Managers](#14-files-resources-and-context-managers)
15. [Iterators, Generators, and Laziness](#15-iterators-generators-and-laziness)
16. [Common Pitfalls](#16-common-pitfalls)
17. [Performance Notes](#17-performance-notes)
18. [Testing Conventions](#18-testing-conventions)
19. [Project Layout and Tooling](#19-project-layout-and-tooling)
20. [Quick Reference Checklist](#20-quick-reference-checklist)

---

## 1. How to Read This Document

Python's conventions come from three sources, in descending order of authority:

| Source | What it governs |
|---|---|
| **The language reference** | What is legal — syntax, semantics, evaluation order |
| **PEP 8** | Style — layout, naming, whitespace |
| **PEP 20 (Zen of Python)** | Judgment — what to prefer when several legal styles exist |

Additional relevant PEPs: **PEP 257** (docstrings), **PEP 484/585/604** (type hints), **PEP 3107** (annotations).

Style rules exist to reduce the reader's effort. When a rule would make a specific piece of code *less* readable, PEP 8 itself says to break it. Consistency within a file beats consistency with the guide; consistency within a project beats consistency within a file.

Throughout, examples are marked:

- ✅ preferred
- ⚠️ legal but problematic
- ❌ wrong or broken

---

## 2. Lexical Structure

### 2.1 Indentation Is Syntax

Python delimits blocks by indentation, not braces. The indentation level is part of the grammar, and inconsistent indentation is a syntax error, not a formatting complaint.

### Notebook setup

Run this cell first. The examples that read and write files do so in a
throwaway scratch directory, so nothing here touches your own files.

In [ ]:
import os
import tempfile

_scratch = tempfile.mkdtemp(prefix="python-style-")
os.chdir(_scratch)
print("Working directory for the file examples:", _scratch)

In [ ]:
condition = True


def do_this():
    print("this")


def do_that():
    print("that")


def do_this_always():
    print("always")


if condition:
    do_this()
    do_that()
do_this_always()

Rules:

- **Four spaces per level.** Never tabs. Python 3 raises `TabError` when tabs and spaces are mixed inconsistently.
- A block must contain at least one statement. Use `pass` as an explicit placeholder:

In [ ]:
def not_implemented_yet():
    pass

- Configure your editor to insert spaces when Tab is pressed and to show whitespace characters.

### 2.2 Statements and Line Structure

One statement per line. A newline terminates a statement.

In [ ]:
# ❌ Legal but discouraged
x = 1; y = 2

# ✅
x = 1
y = 2

**Implicit line continuation** — inside `()`, `[]`, or `{}`, a statement may span lines. This is the preferred way to break long lines.

In [ ]:
first_component, second_component, third_component = 1, 2, 3


def some_function(a, b, c):
    return f"{a}-{b}-{c}"


argument_one, argument_two, argument_three = "x", "y", "z"

total = (first_component
         + second_component
         + third_component)

result = some_function(
    argument_one,
    argument_two,
    argument_three,
)

print(total, result)

**Explicit continuation** with a backslash works but is fragile — trailing whitespace after the backslash is a syntax error, and it is easy to miss visually. Reserve it for cases with no brackets available:

In [ ]:
first_component, second_component = 1, 2

# ⚠️ Avoid where parentheses would work
total = first_component \
      + second_component

print(total)

### 2.3 Comments

In [ ]:
def compute():
    return 42


# A block comment describing the code below.
# It aligns with the code it documents.

x = compute()  # An inline comment, two spaces before the hash.
print(x)

- Comments start with `# ` (hash, single space).
- Block comments are indented to the level of the code they describe.
- Inline comments are separated from code by at least two spaces. Use them sparingly — they crowd the line.
- Keep comments accurate. **A wrong comment is worse than no comment**, because readers trust it.

### 2.4 String Literals

In [ ]:
name = "Ada"
age = 36

single = 'text'
double = "text"
triple = """spans
multiple lines"""

raw = r"C:\Users\name"          # backslashes are literal - use for regex and paths
formatted = f"{name} is {age}"   # f-string
byte_string = b"raw bytes"

print(single, double, repr(triple), raw, formatted, byte_string)

Conventions:

- Pick one quote style per project and stick to it. `black` normalizes to double quotes; that is the de facto default.
- Use the other style to avoid escaping: `"it's fine"` rather than `'it\'s fine'`.
- Use raw strings for regular expressions, always: `re.compile(r"\d+")`.
- Adjacent string literals concatenate at compile time — useful for long messages:

In [ ]:
message = (
    "This is a long message that would exceed the line limit, "
    "so it is split across several adjacent literals."
)

### 2.5 String Formatting: Which to Use

In [ ]:
name, count = "Ada", 3

f"{name} has {count} items"                    # ✅ default choice
"{} has {} items".format(name, count)          # legacy, still fine
"%s has %d items" % (name, count)              # oldest; avoid in new code

f-strings are fastest and most readable. Two exceptions:

1. **Logging** — pass the format arguments to the logger so formatting is skipped when the level is disabled:

In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
logger = logging.getLogger("demo")


def expensive_repr(o):
    print("(expensive_repr was called)")
    return repr(o)


obj = {"rows": 3}

# ❌ Formats even when DEBUG is off — expensive_repr runs regardless
logger.debug(f"Processing {expensive_repr(obj)}")

# ✅ Deferred formatting — the logger skips it entirely at this level
logger.debug("Processing %s", obj)

2. **Template strings from untrusted input** — never `eval`-adjacent formatting on user data.

Useful f-string features:

In [ ]:
value = 3.14159
f"{value:.2f}"        # '3.14'
f"{value:>10.2f}"     # '      3.14'  (right-aligned, width 10)
f"{1234567:,}"        # '1,234,567'
f"{0.756:.1%}"        # '75.6%'
f"{value=}"           # 'value=3.14159'  (debugging, Python 3.8+)
f"{name!r}"           # repr instead of str

### 2.6 Numeric Literals

In [ ]:
1_000_000        # underscores as digit separators
0b1010           # binary
0o755            # octal
0xFF             # hexadecimal
1e-3             # float
3 + 4j           # complex

---

## 3. Names and Naming Conventions

### 3.1 The Table

| Kind | Convention | Example |
|---|---|---|
| Module / package | `lowercase`, short, no underscores if possible | `csv`, `dataloader` |
| Class / Exception | `CapWords` | `HttpClient`, `ParseError` |
| Function / method | `snake_case` | `parse_header()` |
| Variable / argument | `snake_case` | `row_count` |
| Constant | `UPPER_SNAKE_CASE` | `MAX_RETRIES` |
| Type variable | `CapWords`, short | `T`, `KT`, `AnyStr` |
| Internal use | leading underscore | `_cache`, `_helper()` |
| Name-mangled | two leading underscores | `__slot` |
| Dunder | two leading and trailing | `__init__` — never invent your own |

### 3.2 Underscore Semantics

In [ ]:
def compute():
    return 0


_internal = 1        # "private by convention"; not imported by `from m import *`
__mangled = 2        # inside a class, becomes _ClassName__mangled
__dunder__ = 3       # reserved by the language - do not create new ones
class_ = 4           # trailing underscore avoids clashing with a keyword
_ = compute()        # conventional throwaway name

print(_internal, __mangled, __dunder__, class_, _)

Name mangling exists to prevent accidental attribute collisions in subclasses, not to enforce privacy. Python has no true private members; a single leading underscore is the honest signal and the one you should reach for.

### 3.3 Choosing Good Names

In [ ]:
from typing import NamedTuple


class Record(NamedTuple):
    student_id: str
    score: int


records = [Record("s1", 91), Record("s2", 65), Record("s3", 78)]
passing_threshold = 70

# ❌ Cryptic — the reader has to decode every name and index
x = [(1, "s1", 91), (0, "s2", 65), (1, "s3", 78)]
n = 0
d = {}
for i in x:
    if i[0] > n:
        d[i[1]] = i[2]

# ✅ Says what it means
scores_by_student = {}
for record in records:
    if record.score > passing_threshold:
        scores_by_student[record.student_id] = record.score

print(d)
print(scores_by_student)

Guidelines:

- Length should scale with scope. `i` is fine as a loop index in three lines; a module-level `i` is not.
- Name for meaning, not type: `students`, not `student_list`. The type is visible; the meaning is not.
- Avoid the single-character names `l`, `O`, and `I` — indistinguishable from `1` and `0` in many fonts.
- Booleans read as predicates: `is_valid`, `has_children`, `should_retry`.
- Functions that do something are verbs (`fetch_rows`); functions that answer something are questions (`is_empty`).
- Do not shadow builtins: `list`, `dict`, `type`, `id`, `sum`, `max`, `input`, `str`, `file`, `next`, `filter`, `object`, `bytes`, `hash`. Append an underscore or rename.

In [ ]:
# ❌ Breaks list() for the rest of the scope
list = [1, 2, 3]
try:
    list(range(3))
except TypeError as exc:
    print(f"TypeError: {exc}")

del list          # restore the builtin so the rest of this notebook still runs
print(list(range(3)))

# ✅ Name it for what it holds instead
values = [1, 2, 3]
print(values)

---

## 4. Layout and Whitespace

### 4.1 Line Length

PEP 8 says 79 characters; many projects use 88 (`black`'s default) or 100. Pick one, record it in your config, and let a formatter enforce it. Docstrings and comments should wrap at 72 characters — narrow columns are easier to read as prose.

### 4.2 Blank Lines

In [ ]:
import os


CONSTANT = 1


class Example:
    """Two blank lines before top-level definitions."""

    def method_one(self):
        """One blank line between methods."""
        ...

    def method_two(self):
        ...


def top_level_function():
    ...

- Two blank lines around top-level functions and classes.
- One blank line between methods.
- Single blank lines inside functions to separate logical steps — sparingly. If a function needs many, it probably wants to be several functions.

### 4.3 Whitespace in Expressions

In [ ]:
def spam(a, b):
    return a, b


def foo(a, b):
    return a + b


ham = [0, 1, 2]
eggs = "eggs"

# ✅
spam(ham[1], {eggs: 2})
x = 1
y = x + 1
foo(x, y)
print(x, end="")            # no spaces around = for keyword arguments


def f(x: int = 0) -> int:   # but DO use spaces when annotated
    return x


print()

# ❌ Legal, but noisy — every one of these is a PEP 8 violation
spam( ham[ 1 ], { eggs: 2 } )
x             = 1
y = x+1
foo (x, y)
print(x, end = "")
print()

Around operators, use a single space. It is acceptable to group by precedence when it clarifies:

In [ ]:
# ✅ both acceptable
hypot = x*x + y*y
hypot = x * x + y * y

Slices are treated like a binary operator with equal space on both sides, and colons in a slice normally get none:

In [ ]:
ham = [i for i in range(20)]
lower, upper, offset, x = 2, 9, 1, 3


def upper_fn(v):
    return v + 5


def step_fn(v):
    return 2


print(ham[1:9])
print(ham[lower:upper])
print(ham[lower + offset : upper + offset])   # spaces when the operands are expressions
print(ham[: upper_fn(x) : step_fn(x)])
print(ham[::2])

### 4.4 Trailing Commas

Add a trailing comma when each element is on its own line — it makes diffs one-line-per-change and prevents the classic missing-comma bug.

In [ ]:
# ✅
FLAGS = [
    "verbose",
    "dry_run",
    "force",
]

# ❌ A silent string-concatenation bug
NAMES = [
    "alpha",
    "beta"      # missing comma
    "gamma",    # -> "betagamma"
]

Do **not** put a trailing comma after `*args`/`**kwargs` in a signature, and note that a trailing comma in a single-element parenthesized expression makes a tuple: `x = (1,)` is a tuple, `x = (1)` is an int.

---

## 5. Expressions and Operators

### 5.1 Comparison Rules

```python
# Identity vs. equality
if value is None:          # ✅ singletons compared with `is`
if value == None:          # ❌

if x is True:              # ⚠️ almost always wrong
if x:                      # ✅ truthiness

if not items:              # ✅ empty check
if len(items) == 0:        # ⚠️ verbose

if isinstance(x, int):     # ✅ type check
if type(x) == int:         # ⚠️ rejects subclasses
```

*(Kept as a listing rather than a runnable cell: these are bare `if` clauses
shown side by side for comparison, so they have no bodies to execute.)*

`is` compares object identity. It is correct only for `None`, `True`, `False`, and sentinels you created. Using it on numbers or strings appears to work because of interning and then fails on values outside the cached range:

```python
>>> a = 256; b = 256; a is b
True
>>> a = 257; b = 257; a is b
False       # depends on the interpreter — never rely on this
```

### 5.2 Truthiness

Falsy: `False`, `None`, `0`, `0.0`, `0j`, `""`, `()`, `[]`, `{}`, `set()`, `range(0)`, and any object whose `__bool__` returns `False` or whose `__len__` returns `0`.

Everything else is truthy. Beware the difference between "empty" and "missing":

In [ ]:
config = {"timeout": 0}   # 0 is a legitimate, deliberate value

# ⚠️ Treats 0, "", and [] the same as absent
value = config.get("timeout") or 30
print("with `or`:", value)        # 30 - the caller's 0 was silently discarded

# ✅ Distinguishes absent from a legitimate falsy value
value = config.get("timeout")
if value is None:
    value = 30
print("with `is None`:", value)   # 0 - preserved

### 5.3 Chaining and Boolean Operators

In [ ]:
x = 42

print(0 <= x < 100)             # ✅ chained comparison
print(0 <= x and x < 100)       # verbose equivalent

# `and`/`or` return an operand, not a bool
supplied_name = ""
name = supplied_name or "anonymous"

mapping = {"key": "value"}
first_match = mapping and mapping.get("key")

print(name, first_match)

Use `and`/`or`/`not`, never `&`/`|`/`~` for booleans — the latter are bitwise, bind tighter, and do not short-circuit. (In pandas and NumPy the opposite holds for element-wise arrays; that is a library convention, not a language one.)

### 5.4 The Walrus Operator

Assignment expressions (`:=`, Python 3.8+) bind a name inside an expression. Use them to avoid computing something twice or to tighten a loop:

In [ ]:
import io
import re

pattern = re.compile(r"id=(\d+)")
line = "user id=42"

# ✅ Clear win — the match is computed once and reused
if (match := pattern.search(line)) is not None:
    print(match.group(1))


def process(chunk):
    pass


stream = io.StringIO("x" * 20_000)
while (chunk := stream.read(8192)):
    process(chunk)


def f(v):
    return v + 1


x = 1

# ⚠️ Clever but harder to read
print([y := f(x), y ** 2, y ** 3])

### 5.5 Conditional Expressions

In [ ]:
n = 7
label = "even" if n % 2 == 0 else "odd"      # ✅ short and clear
print(label)

Do not nest them. If you need two conditions, write an `if`/`elif` block.

### 5.6 Operator Precedence Traps

In [ ]:
a, b, c = 1, 1, [1, 2]

# Unary minus binds tighter than **? No - ** binds tighter than unary minus:
print(-2 ** 2)        # -4, not 4

# `not` binds loosely
print(not a == b)     # means not (a == b)  ✅ but write a != b

# `in` and comparison chaining
print(a == b in c)    # means (a == b) and (b in c) - surprising; parenthesize

When in doubt, add parentheses. They cost nothing and remove a class of bug.

---

## 6. Statements and Control Flow

### 6.1 Conditionals

In [ ]:
score = 87

if score >= 90:
    grade = "A"
elif score >= 80:
    grade = "B"
else:
    grade = "F"

print(grade)

Prefer flat code to nested code. **Guard clauses** — early returns for the exceptional cases — keep the main path at the left margin:

In [ ]:
# ⚠️ Arrow-shaped
def process(order):
    if order is not None:
        if order.is_valid():
            if not order.is_shipped():
                return ship(order)
            else:
                return None
    return None

# ✅ Flat
def process(order):
    if order is None:
        return None
    if not order.is_valid():
        return None
    if order.is_shipped():
        return None
    return ship(order)

For dispatch on a value, a dictionary often beats a long `elif` chain:

In [ ]:
def load_csv(path):
    return f"loaded {path} as csv"


def load_json(path):
    return f"loaded {path} as json"


def load_parquet(path):
    return f"loaded {path} as parquet"


HANDLERS = {
    "csv": load_csv,
    "json": load_json,
    "parquet": load_parquet,
}


def load(path, extension):
    handler = HANDLERS.get(extension)
    if handler is None:
        raise ValueError(f"Unsupported format: {extension!r}")
    return handler(path)


print(load("scores.csv", "csv"))

### 6.2 Structural Pattern Matching (3.10+)

`match` is not a switch statement — it destructures.

In [ ]:
from dataclasses import dataclass


@dataclass
class Point:
    x: int
    y: int


MODIFIERS = {"shift", "ctrl"}


def handle_click(x, y):
    print("click at", x, y)


def handle_modifier(code):
    print("modifier", code)


def handle_origin():
    print("origin")


def handle_sequence(first, rest):
    print("sequence", first, rest)


def dispatch(event):
    match event:
        case {"type": "click", "pos": (x, y)}:
            handle_click(x, y)
        case {"type": "key", "code": code} if code in MODIFIERS:
            handle_modifier(code)
        case Point(x=0, y=0):
            handle_origin()
        case [first, *rest]:
            handle_sequence(first, rest)
        case _:
            raise ValueError(f"Unknown event: {event!r}")


dispatch({"type": "click", "pos": (3, 4)})
dispatch({"type": "key", "code": "shift"})
dispatch(Point(0, 0))
dispatch([1, 2, 3])

Pitfall: a bare name in a pattern **binds**, it does not compare.

```python
RED = 1

match color:
    case RED:        # ❌ binds `RED` to color; matches everything
        ...
    case Color.RED:  # ✅ dotted names compare
        ...
```

*(Kept as a listing: Python refuses to compile this - `SyntaxError: name
capture 'RED' makes remaining patterns unreachable` - which is exactly the
point being made.)*

Use `match` for genuinely structural data (parsed messages, ASTs, tagged unions). For simple value dispatch, a dict is clearer.

### 6.3 Loops

Iterate over objects directly. Indexing by range is almost always a translation from another language.

In [ ]:
from dataclasses import dataclass


@dataclass
class Item:
    name: str
    score: int


items = [Item("a", 3), Item("b", 1), Item("c", 2)]
names = ["Ada", "Grace"]
scores = [91, 95]

# ❌ Indexing by range — almost always a translation from another language
for i in range(len(items)):
    print(items[i])

# ✅ Iterate over the objects directly
for item in items:
    print(item)

# ✅ when the index is genuinely needed
for index, item in enumerate(items):
    print(index, item)

# ✅ parallel iteration
for name, score in zip(names, scores, strict=True):   # strict= is 3.10+
    print(name, score)

# ✅ reverse, sorted
for item in reversed(items):
    print(item)
for item in sorted(items, key=lambda r: r.score, reverse=True):
    print(item)

`zip` silently truncates to the shortest input; `strict=True` raises instead, which is what you usually want.

### 6.4 `break`, `continue`, and `for ... else`

The `else` clause on a loop runs when the loop finished **without** `break`. It is genuinely useful for search loops but widely misread — comment it or avoid it.

In [ ]:
from dataclasses import dataclass


@dataclass
class Candidate:
    name: str

    def matches(self, query):
        return self.name == query


candidates = [Candidate("alpha"), Candidate("beta")]
query = "beta"

for candidate in candidates:
    if candidate.matches(query):
        result = candidate
        break
else:
    raise LookupError("no candidate matched")

print(result)

### 6.5 Modifying While Iterating

Mutating a collection during iteration over it is undefined behavior in practice — items get skipped, or `RuntimeError` is raised.

In [ ]:
from dataclasses import dataclass


@dataclass
class Item:
    name: str
    expired: bool


def fresh_items():
    return [Item("a", True), Item("b", False), Item("c", True)]


# ❌ Skips elements — the loop index advances while the list shrinks
items = fresh_items()
for item in items:
    if item.expired:
        items.remove(item)
print("mutated while iterating:", items)   # the second expired item survives

# ✅ Build a new list
items = fresh_items()
items = [item for item in items if not item.expired]
print("rebuilt:", items)

# ✅ Or iterate over a copy when in-place mutation is required
items = fresh_items()
for item in list(items):
    if item.expired:
        items.remove(item)
print("copy-iterated:", items)

The same applies to dictionaries: `for key in dict(d):` or `for key in list(d.keys()):` if you will delete keys.

---

## 7. Functions

### 7.1 Signatures

In [ ]:
class Response:
    """Stand-in for a real HTTP response type."""


def fetch(
    url: str,
    *,
    timeout: float = 10.0,
    retries: int = 3,
    headers: dict[str, str] | None = None,
) -> Response:
    ...


import inspect

print(inspect.signature(fetch))

- Everything after a bare `*` is **keyword-only**. Use this for options and flags — call sites become self-documenting, and you can reorder parameters later without breaking callers.
- Everything before a `/` is positional-only (3.8+). Useful for library APIs where you do not want parameter names to become part of the contract.
- Keep parameter counts low. More than about five suggests a missing dataclass.

In [ ]:
def create_user(first, last, admin=False, verified=False, active=True, age=None):
    return f"{first} {last} admin={admin} verified={verified} active={active} age={age}"


# ⚠️ Unreadable call site — what do the three booleans mean?
print(create_user("Ada", "Lovelace", True, False, True, 30))

# ✅ Self-documenting
print(create_user("Ada", "Lovelace", admin=True, verified=False, active=True, age=30))

### 7.2 The Mutable Default Argument

This is the single most famous Python pitfall. Default values are evaluated **once**, at function definition, and shared across every call.

In [ ]:
# ❌ The list persists between calls
def append_to(item, target=[]):
    target.append(item)
    return target

append_to(1)   # [1]
append_to(2)   # [1, 2]  — surprise

# ✅
def append_to(item, target=None):
    if target is None:
        target = []
    target.append(item)
    return target

The same applies to `{}`, `set()`, and to any expression evaluated at definition time — `datetime.now()` as a default captures import time, not call time.

### 7.3 `*args` and `**kwargs`

In [ ]:
def log(level, *messages, **context):
    text = " ".join(str(m) for m in messages)
    print(f"[{level}] {text} {context}")

log("INFO", "loaded", "3 rows", source="db")

Unpacking at the call site:

In [ ]:
def run(*args, **kwargs):
    print("args:", args, "kwargs:", kwargs)


args = (1, 2)
opts = {"verbose": True}
run(*args, **opts)

Use `*args`/`**kwargs` for genuine pass-through (decorators, wrappers). Do not use them to avoid writing a signature — they erase the API from the reader and from tooling.

### 7.4 Return Values

In [ ]:
from dataclasses import dataclass


@dataclass
class User:
    user_id: int
    name: str


USERS = {1: User(1, "Ada")}


# ✅ Consistent return types
def find_user(user_id: int) -> User | None:
    return USERS.get(user_id)


# ⚠️ Returns bool sometimes, str other times — callers must guess
def check(x):
    if x > 0:
        return "positive"
    return False


print(find_user(1), find_user(99))
print(check(5), check(-5))

A function that returns `None` on one path should return `None` explicitly on all of them, or raise. Returning multiple values is fine as a tuple, but past two or three, return a `NamedTuple` or dataclass so fields have names.

In [ ]:
from statistics import mean, median
from typing import NamedTuple


class Stats(NamedTuple):
    mean: float
    median: float
    count: int


def summarize(values) -> Stats:
    return Stats(mean(values), median(values), len(values))


data = [88, 92, 79, 95]
result = summarize(data)
print(result.mean)          # readable at the call site
print(result)

### 7.5 Function Size and Purity

- A function should do one thing and fit on a screen. If you need a comment to mark sections inside it, those sections are functions.
- Prefer **pure** functions — output determined by input, no side effects. They are trivially testable.
- Separate computation from I/O. `load()`, `compute()`, `save()` is testable; a single `do_everything()` that reads a file, transforms, and prints is not.

### 7.6 Lambdas

In [ ]:
from dataclasses import dataclass


@dataclass
class Person:
    first: str
    last: str


rows = [Person("Grace", "Hopper"), Person("Ada", "Lovelace")]

# ✅ Small inline key function
rows.sort(key=lambda r: (r.last, r.first))
print(rows)

# ❌ Never bind a lambda to a name — just use def
square = lambda x: x ** 2


# ✅
def square(x):
    return x ** 2


print(square(4))

A named `def` gets a real `__name__` for tracebacks, can hold a docstring, and can be annotated.

### 7.7 Decorators

In [ ]:
import functools

def retry(times: int = 3):
    def decorator(func):
        @functools.wraps(func)          # preserves __name__, __doc__, signature
        def wrapper(*args, **kwargs):
            for attempt in range(times):
                try:
                    return func(*args, **kwargs)
                except TransientError:
                    if attempt == times - 1:
                        raise
        return wrapper
    return decorator


@retry(times=5)
def fetch_rows(): ...

Always apply `functools.wraps`. Without it the wrapped function loses its name and docstring, and debugging becomes guesswork.

Standard-library decorators worth knowing: `functools.cache` / `lru_cache`, `functools.singledispatch`, `property`, `staticmethod`, `classmethod`, `contextlib.contextmanager`, `dataclasses.dataclass`.

### 7.8 Closures in Loops

In [ ]:
# ❌ All three print 2 — `i` is looked up when called, not when defined
callbacks = []
for i in range(3):
    callbacks.append(lambda: print(i))

# ✅ Bind the value with a default argument
for i in range(3):
    callbacks.append(lambda i=i: print(i))

# ✅ Or with a factory
def make_callback(i):
    return lambda: print(i)

---

## 8. Data Structures and Idioms

### 8.1 Choosing a Type

| Need | Use | Lookup cost |
|---|---|---|
| Ordered, changing sequence | `list` | O(n) search |
| Fixed record / hashable sequence | `tuple` | O(n) search |
| Key → value | `dict` | O(1) |
| Membership, deduplication | `set` | O(1) |
| Queue / stack from both ends | `collections.deque` | O(1) at both ends |
| Counting | `collections.Counter` | O(1) |
| Grouped accumulation | `collections.defaultdict` | O(1) |
| Numeric array math | `numpy.ndarray` | vectorized |

The most common performance mistake in Python is repeated `in` on a list where a set was appropriate:

In [ ]:
from dataclasses import dataclass


@dataclass
class Row:
    id: int


rows = [Row(1), Row(2), Row(1), Row(3)]

# ❌ O(n) per check → O(n·m) overall
seen = []
for row in rows:
    if row.id not in seen:
        seen.append(row.id)
print(seen)

# ✅ O(1) per check
seen = set()
for row in rows:
    if row.id not in seen:
        seen.add(row.id)
print(sorted(seen))

### 8.2 Comprehensions

In [ ]:
numbers = [1, 2, 3, 4, 5, 6]
names = ["Ada", "Grace", "Katherine"]
words = ["The", "the", "AND", "and"]

squares = [n ** 2 for n in range(10)]
evens = [n for n in numbers if n % 2 == 0]
pairs = {name: len(name) for name in names}
uniques = {word.lower() for word in words}
lazy = (n ** 2 for n in range(10 ** 9))      # generator - no memory blowup

print(squares)
print(evens)
print(pairs)
print(sorted(uniques))
print(lazy)                                  # nothing computed yet

Nested loops read in the same order as the equivalent `for` statements:

In [ ]:
matrix = [[1, 2], [3, 4], [5, 6]]

flat = [item for row in matrix for item in row]

# equivalent to
flat_loop = []
for row in matrix:
    for item in row:
        flat_loop.append(item)

print(flat)
print(flat_loop)

When to stop: a comprehension with more than one `for` plus a condition, or one that no longer fits on two lines, should be a loop. Comprehensions are for *building a collection*; using one for side effects is a misuse:

In [ ]:
items = ["a", "b", "c"]

# ❌ Builds and discards a list of None
[print(x) for x in items]

# ✅ Say what you mean
for x in items:
    print(x)

### 8.3 Dictionary Idioms

In [ ]:
from collections import defaultdict
from dataclasses import dataclass


@dataclass
class Record:
    category: str
    value: int


config = {"key": "value"}
default = "fallback"
x = 1
records = [Record("a", 1), Record("b", 2), Record("a", 3)]
defaults = {"debug": False, "level": 1}
overrides = {"level": 2}
d = {"one": 1, "two": 2}

print(config.get("key"))                    # None if missing
print(config.get("missing", default))       # default if missing
config.setdefault("items", []).append(x)
print(config)

groups = defaultdict(list)
for record in records:
    groups[record.category].append(record)
print(dict(groups))

merged = {**defaults, **overrides}   # or defaults | overrides  (3.9+)
print(merged)

for key, value in d.items():         # ✅
    print(key, value)
for key in d:                        # iterates keys
    print(key)

Note that `defaultdict` inserts on read — `groups["missing"]` creates an entry. Use `.get()` when you only want to inspect.

### 8.4 Unpacking

In [ ]:
a, b = 1, 2
values = [10, 20, 30, 40]
points = [(1, 2), (3, 4)]

a, b = b, a                            # swap, no temporary
first, *rest = [1, 2, 3, 4]            # first=1, rest=[2, 3, 4]
first_v, *_, last = values             # discard the middle
(x, y), (z, w) = points                # nested


def f():
    return 1, 2


c, d_ = f()

print(a, b)
print(first, rest)
print(first_v, last)
print(x, y, z, w)
print(c, d_)

### 8.5 Sorting

In [ ]:
from dataclasses import dataclass
from operator import attrgetter, itemgetter


@dataclass
class Row:
    name: str
    score: int


rows = [Row("Ada", 91), Row("Grace", 95), Row("Katherine", 91)]
pairs = [("a", 2), ("b", 1)]

print(sorted(rows, key=lambda r: r.score))              # new list
print(rows.sort(key=lambda r: r.score))                 # in place, returns None
print(sorted(rows, key=lambda r: (-r.score, r.name)))   # multi-key, mixed direction

print(sorted(rows, key=attrgetter("score")))            # faster than a lambda
print(sorted(pairs, key=itemgetter(1)))

Python's sort is **stable**: equal elements keep their relative order. That lets you sort by successive keys in reverse order of importance to get a multi-key sort.

Pitfall: `rows = rows.sort()` sets `rows` to `None`. In-place methods (`sort`, `reverse`, `append`, `extend`, `update`) return `None` by design so you cannot chain them accidentally.

### 8.6 Copying

In [ ]:
import copy

original = [[1, 2], [3, 4]]

shallow = original[:]              # or list(original), original.copy()
deep = copy.deepcopy(original)     # recursive

original[0].append(99)
print("shallow sees the change:", shallow)
print("deep does not:        ", deep)

A shallow copy duplicates the container but shares the contained objects:

In [ ]:
grid = [[0] * 3] * 3        # ❌ three references to the SAME row
grid[0][0] = 1              # -> [[1,0,0], [1,0,0], [1,0,0]]

grid = [[0] * 3 for _ in range(3)]   # ✅ three distinct rows

### 8.7 Strings

In [ ]:
parts = ["a", "b", "c"]
name, other = "Ada", "ADA"

print("".join(parts))                    # ✅ O(n) concatenation

text = ""
for part in parts:                       # ❌ O(n²) — each += copies the whole string
    text += part
print(text)

print("a,b,c".split(","))
print("  text  ".strip())
print("path/to/file".removeprefix("path/"))     # 3.9+
print(name.casefold() == other.casefold())      # case-insensitive comparison

Strings are immutable: every `+=` allocates a new object. Build a list and `join` it.

---

## 9. Classes and Object Model

### 9.1 A Well-Formed Class

In [ ]:
class Account:
    """A bank account with a running balance.

    Attributes:
        owner: Name of the account holder.
        balance: Current balance in dollars.
    """

    MINIMUM_BALANCE = 0.0          # class attribute (shared)

    def __init__(self, owner: str, balance: float = 0.0) -> None:
        self.owner = owner         # instance attributes
        self._balance = balance    # internal by convention

    @property
    def balance(self) -> float:
        """Current balance; read-only from outside."""
        return self._balance

    def deposit(self, amount: float) -> None:
        if amount <= 0:
            raise ValueError(f"deposit must be positive, got {amount}")
        self._balance += amount

    def __repr__(self) -> str:
        return f"Account(owner={self.owner!r}, balance={self._balance})"

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Account):
            return NotImplemented
        return (self.owner, self._balance) == (other.owner, other._balance)

Conventions:

- `self` is the first parameter of every instance method. It is a convention, not a keyword, but never rename it.
- Define `__repr__` on every class you will debug. It should be unambiguous and, ideally, `eval`-able. `__str__` is for end users and falls back to `__repr__`.
- If you define `__eq__`, define `__hash__` too, or the class becomes unhashable. Setting `__eq__` sets `__hash__ = None` implicitly.

### 9.2 Class vs. Instance Attributes

In [ ]:
class Dog:
    tricks = []              # ❌ shared by ALL instances

    def add(self, trick):
        self.tricks.append(trick)

a, b = Dog(), Dog()
a.add("roll over")
b.tricks                     # ['roll over'] — leaked

class Dog:
    def __init__(self):
        self.tricks = []     # ✅ per instance

Class-level mutable state is the object-oriented twin of the mutable-default pitfall. Class attributes are for constants and defaults that are immutable.

### 9.3 Dataclasses

For classes that mostly hold data, `@dataclass` writes `__init__`, `__repr__`, and `__eq__` for you:

In [ ]:
from dataclasses import dataclass, field

@dataclass(frozen=True, slots=True)
class Point:
    x: float
    y: float
    label: str = ""
    tags: list[str] = field(default_factory=list)   # ✅ not `= []`

    def distance_to(self, other: "Point") -> float:
        return ((self.x - other.x) ** 2 + (self.y - other.y) ** 2) ** 0.5

- `frozen=True` makes instances immutable and hashable — a good default for value objects.
- `slots=True` (3.10+) reduces memory and blocks typo-attribute creation.
- `field(default_factory=list)` is mandatory for mutable defaults; a bare `= []` raises at class creation.

Alternatives: `typing.NamedTuple` for immutable tuple-like records, `pydantic.BaseModel` when you need validation and parsing.

### 9.4 Properties over Getters and Setters

In [ ]:
# ❌ Java-style
class Circle:
    def get_radius(self): return self._r
    def set_radius(self, v): self._r = v

# ✅ Pythonic — start with a plain attribute
class Circle:
    def __init__(self, radius):
        self.radius = radius

# ✅ Add a property later if validation becomes necessary; callers do not change
class Circle:
    @property
    def radius(self):
        return self._radius

    @radius.setter
    def radius(self, value):
        if value < 0:
            raise ValueError("radius must be non-negative")
        self._radius = value

The point is that Python lets you *upgrade* an attribute to a property without changing the interface, so writing accessors upfront buys nothing.

### 9.5 Inheritance and Composition

In [ ]:
class Base:
    def __init__(self, name):
        self.name = name

class Derived(Base):
    def __init__(self, name, extra):
        super().__init__(name)      # ✅ always super(), never Base.__init__(self, ...)
        self.extra = extra

- Prefer composition to inheritance. Inheritance couples the subclass to the parent's implementation, not just its interface.
- Use `abc.ABC` and `@abstractmethod` when you truly need an interface contract; use `typing.Protocol` for structural typing without a base class.
- Multiple inheritance follows the MRO (`ClassName.__mro__`). It is workable for mixins with cooperative `super()` calls and a nightmare otherwise.

### 9.6 Static and Class Methods

In [ ]:
class Temperature:
    def __init__(self, celsius): self.celsius = celsius

    @classmethod
    def from_fahrenheit(cls, f):        # alternative constructor
        return cls((f - 32) * 5 / 9)

    @staticmethod
    def is_valid(celsius):              # no access to cls or self
        return celsius >= -273.15

If a `@staticmethod` never touches the class conceptually, it probably belongs at module level as a plain function.

---

## 10. Modules, Packages, and Imports

### 10.1 Import Order

Three groups, separated by blank lines, alphabetized within each:

```python
# 1. Standard library
import json
import os
from pathlib import Path

# 2. Third party
import numpy as np
import pandas as pd
import requests

# 3. Local application
from myproject.config import settings
from myproject.utils import normalize
```

*(Kept as a listing: it names a `myproject` package that does not exist here.)*

`isort` or `ruff --select I` maintains this automatically.

### 10.2 Import Style

```python
import os                              # ✅
from pathlib import Path               # ✅ specific name
import numpy as np                     # ✅ conventional alias
from mypackage import module           # ✅

from os import *                       # ❌ pollutes the namespace, hides origins
import os, sys                         # ❌ one import per line
from ..deeply.nested import thing      # ⚠️ relative imports: keep shallow
```

*(Kept as a listing: the discouraged forms would either fail to import here or
wreck the namespace for every cell below.)*

Wildcard imports make it impossible to tell where a name came from and break linters. The one tolerable use is in a package `__init__.py` re-exporting a curated `__all__`.

### 10.3 Absolute vs. Relative

```python
from myproject.data import loader      # ✅ absolute — explicit, refactor-friendly
from .data import loader               # acceptable inside a package
from ..core import engine              # ⚠️ two levels up is a design smell
```

*(Kept as a listing: relative imports are only meaningful inside a package,
not in a notebook.)*

### 10.4 The `__main__` Guard

```python
def main() -> int:
    ...
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
```

*(Kept as a listing: a notebook's `__name__` is already `"__main__"`, so running
this would raise `SystemExit` and stop the kernel.)*

Without this guard, importing the module executes its top level. With it, the file works both as a library and as a script. Returning an exit code through `SystemExit` gives the shell something to test.

Keep module top level free of side effects: no network calls, no file reads, no prints at import time.

### 10.5 Circular Imports

Symptom: `ImportError: cannot import name X from partially initialized module`.

Fixes, in order of preference:

1. Extract the shared code into a third module that both import.
2. Move the import inside the function that needs it (deferred import).
3. Use `if typing.TYPE_CHECKING:` when the import exists only for annotations:

In [ ]:
from typing import TYPE_CHECKING

if TYPE_CHECKING:
    from myproject.models import User

def greet(user: "User") -> str:
    return f"Hello, {user.name}"

---

## 11. Errors and Exceptions

### 11.1 EAFP over LBYL

Python favors *Easier to Ask Forgiveness than Permission*:

In [ ]:
import os

path = "does-not-exist.txt"

# ⚠️ LBYL — racy and slower on the happy path
if os.path.exists(path):
    with open(path) as f:          # file may vanish between the two lines
        data = f.read()
else:
    data = ""
print("LBYL:", repr(data))

# ✅ EAFP
try:
    with open(path) as f:
        data = f.read()
except FileNotFoundError:
    data = ""
print("EAFP:", repr(data))

LBYL is still right when the check is cheap and the failure is expected and frequent.

### 11.2 Catching Correctly

```python
# ❌ Swallows everything, including KeyboardInterrupt and typos in your own code
try:
    result = compute()
except:
    pass

# ❌ Still too broad, and silent
except Exception:
    pass

# ✅ Specific, and it says what happened
try:
    result = compute()
except (ValueError, TypeError) as exc:
    logger.warning("compute failed for %r: %s", payload, exc)
    result = FALLBACK
```

*(Kept as a listing: the middle fragment is a deliberately orphaned `except`
clause, which is a `SyntaxError` on its own.)*

Rules:

- Catch the narrowest exception that could actually occur.
- Never write a bare `except:`. It catches `SystemExit` and `KeyboardInterrupt` and makes the program unkillable.
- Never silently `pass`. If you genuinely intend to ignore something, say so: `except FileNotFoundError: pass  # optional cache file`, or use `contextlib.suppress(FileNotFoundError)`.
- Keep the `try` block small — wrap only the line that can fail, so you do not accidentally catch an error from unrelated code.

### 11.3 Raising

In [ ]:
timeout = -1

# ✅ Specific type, actionable message including the offending value
try:
    raise ValueError(f"expected a positive timeout, got {timeout!r}")
except ValueError as exc:
    print(f"{type(exc).__name__}: {exc}")

# ❌ Loses all information
try:
    raise Exception("error")
except Exception as exc:
    print(f"{type(exc).__name__}: {exc}")

Choose the right built-in: `ValueError` (right type, wrong value), `TypeError` (wrong type), `KeyError`/`IndexError` (lookup), `FileNotFoundError`, `NotImplementedError` (abstract method), `RuntimeError` (last resort).

Define a project exception hierarchy so callers can catch at the granularity they need:

In [ ]:
class DataPipelineError(Exception):
    """Base class for all errors raised by this package."""

class SchemaError(DataPipelineError): ...
class SourceUnavailableError(DataPipelineError): ...

### 11.4 Chaining and Context

In [ ]:
class SchemaError(Exception):
    pass


def parse(raw):
    raise ValueError(f"could not parse {raw!r}")


raw, path = "not-a-row", "rows.csv"

try:
    try:
        parse(raw)
    except ValueError as exc:
        raise SchemaError(f"invalid row in {path}") from exc     # ✅ preserves the cause
except SchemaError as exc:
    print(f"{type(exc).__name__}: {exc}")
    print("  caused by:", repr(exc.__cause__))

# `raise SchemaError(...) from None` deliberately suppresses the context
try:
    try:
        parse(raw)
    except ValueError:
        raise SchemaError("invalid row") from None
except SchemaError as exc:
    print(f"{type(exc).__name__}: {exc}, cause: {exc.__cause__!r}")

`raise` with no argument inside an `except` block re-raises the current exception with its original traceback — the right way to log and propagate:

```python
except TimeoutError:
    logger.error("timed out talking to %s", host)
    raise
```

*(Kept as a listing: an `except` clause with no `try` is a `SyntaxError`.)*

### 11.5 `else` and `finally`

In [ ]:
class Connection:
    def fetch_all(self):
        return ["row-1", "row-2"]


def connect():
    return Connection()


def cleanup():
    print("cleanup ran")


def load():
    try:
        conn = connect()
    except ConnectionError:
        return None
    else:
        # runs only if no exception - keeps the try block minimal
        return conn.fetch_all()
    finally:
        # always runs, exception or not
        cleanup()


print(load())

Do not `return` inside `finally` — it discards any in-flight exception.

### 11.6 Assertions

In [ ]:
value = 42
assert isinstance(value, int), "internal invariant violated"
print("invariant holds")

`assert` is for programmer errors and invariants, **not** validation of external input — it is stripped entirely when Python runs with `-O`. Never use it to check user data, permissions, or anything security-relevant.

In [ ]:
from dataclasses import dataclass


@dataclass
class User:
    is_admin: bool


user = User(is_admin=False)

# ❌ Vanishes under python -O
try:
    assert user.is_admin, "not authorized"
except AssertionError as exc:
    print(f"AssertionError: {exc}")

# ✅ Survives -O, and is the right exception type
try:
    if not user.is_admin:
        raise PermissionError("not authorized")
except PermissionError as exc:
    print(f"PermissionError: {exc}")

---

## 12. Comments, Docstrings, and Documentation

### 12.1 What to Comment

Comments should explain **why**, not **what**. The code already says what it does.

In [ ]:
import time

i = 0

# ❌ Restates the code
i += 1  # increment i

# ✅ Explains a non-obvious decision
# The vendor API rate-limits at 10 req/s; 0.15s keeps us safely under
# even when the clock drifts on the worker nodes.
time.sleep(0.15)

# ✅ Marks a known compromise
# TODO(pmolnar): replace with the batched endpoint once it leaves beta.

print("i =", i)

Comment: workarounds, non-obvious algorithms, references to external specs or tickets, deliberate deviations from the obvious approach, and units/coordinate conventions. Do not comment: anything a better name would have expressed.

### 12.2 Docstrings (PEP 257)

Every public module, class, function, and method gets a docstring.

```python
def resample(series, freq, method="mean"):
    """Aggregate a time series to a coarser frequency.

    The first line is a one-line summary in the imperative mood, ending
    with a period. A blank line separates it from the body.

    Args:
        series: Time-indexed values to aggregate.
        freq: Pandas offset alias, e.g. "1D" or "1h".
        method: Aggregation to apply. One of "mean", "sum", "max".

    Returns:
        A new series indexed at the requested frequency.

    Raises:
        ValueError: If `method` is not a supported aggregation.

    Example:
        >>> resample(daily_prices, "1W", method="mean")
    """
```

Formatting rules:

- Triple double quotes, always, even for one-liners.
- One-line docstring: summary on the same line as the quotes, closing quotes on the same line.
- Multi-line: summary line, blank line, details, closing quotes on their own line.
- Imperative mood — "Return the sum", not "Returns the sum" (though NumPy style uses the indicative; follow your project).

Three common styles — **Google** (above), **NumPy** (underlined section headers), and **reStructuredText/Sphinx** (`:param x:`). Any is fine; mixing them within a project is not.

### 12.3 Documenting Modules

In [ ]:
"""Loaders for the vendor feed.

This module reads the nightly CSV drops, normalizes column names, and
yields validated records. It does not touch the database; see
`myproject.sink` for persistence.
"""

---

## 13. Type Hints

### 13.1 Basics

Annotations are optional, are **not enforced at run time**, and exist for readers and static checkers (`mypy`, `pyright`, `ruff`).

In [ ]:
def average(values: list[float]) -> float:
    return sum(values) / len(values)

count: int = 0
names: list[str] = []
lookup: dict[str, int] = {}
maybe: str | None = None                  # 3.10+; older: Optional[str]
pair: tuple[int, str] = (1, "a")
row: tuple[int, ...] = (1, 2, 3)          # variable-length homogeneous tuple

Since 3.9 the builtin generics (`list`, `dict`, `set`, `tuple`) are subscriptable directly — no more `typing.List`. Since 3.10, `X | Y` replaces `Union[X, Y]` and `X | None` replaces `Optional[X]`.

### 13.2 Useful Constructs

In [ ]:
from collections.abc import Callable, Iterable, Iterator, Sequence, Mapping
from typing import Any, Literal, Protocol, TypeVar, TypedDict, Final, overload

Handler = Callable[[str, int], bool]
Mode = Literal["r", "w", "a"]
MAX: Final = 100

class Row(TypedDict):
    id: int
    name: str

class SupportsClose(Protocol):
    def close(self) -> None: ...

T = TypeVar("T")
def first(items: Sequence[T]) -> T:
    return items[0]

**Accept broadly, return specifically.** Take `Iterable[str]` or `Sequence[str]` as a parameter; return a concrete `list[str]`.

In [ ]:
from collections.abc import Iterable


# ✅
def normalize(names: Iterable[str]) -> list[str]:
    return [n.strip().lower() for n in names]


print(normalize(("  Ada ", "GRACE", "Katherine  ")))

### 13.3 When to Annotate

- Public functions and class attributes: yes.
- Short internal helpers: optional; annotate when the types are non-obvious.
- `Any` defeats the purpose — use it only at genuine boundaries (deserialization, dynamic plugin loading).

Pitfalls:

- Annotations do not validate. `def f(x: int)` happily accepts a string. Use `pydantic` or explicit checks when validation matters.
- A self-referencing annotation needs quotes or `from __future__ import annotations`:

In [ ]:
from __future__ import annotations       # makes all annotations lazy strings

class Node:
    def add(self, child: Node) -> Node:  # no quotes needed
        ...

---

## 14. Files, Resources, and Context Managers

### 14.1 Always Use `with`

In [ ]:
from pathlib import Path

# Set up the files this example reads (see the setup cell at the top).
Path("data.csv").write_text("id,name\n1,Ada\n", encoding="utf-8")
Path("in.txt").write_text("copied text\n", encoding="utf-8")

# ❌ Leaks the handle if an exception occurs between open() and close()
f = open("data.csv")
data = f.read()
f.close()

# ✅ Closes on the way out, exception or not
with open("data.csv", encoding="utf-8") as f:
    data = f.read()
print(repr(data))

# Multiple resources
with open("in.txt") as src, open("out.txt", "w") as dst:
    dst.write(src.read())

print(Path("out.txt").read_text())

**Always specify `encoding`.** The default depends on the platform locale, so a script that works on macOS may mangle text on Windows. `encoding="utf-8"` is the right default.

Read large files line by line rather than with `.read()`:

In [ ]:
from pathlib import Path

path = "lines.txt"
Path(path).write_text("first\nsecond\nthird\n", encoding="utf-8")


def process(line):
    print("processing:", line)


with open(path, encoding="utf-8") as f:
    for line in f:              # streams; constant memory
        process(line.rstrip("\n"))

### 14.2 `pathlib` over `os.path`

In [ ]:
from pathlib import Path

root = Path("data")
csv_path = root / "raw" / "2026.csv"       # ✅ operator, platform-correct

csv_path.parent.mkdir(parents=True, exist_ok=True)
csv_path.write_text("score\n91\n", encoding="utf-8")

print(csv_path.exists())
print(csv_path.suffix)                     # '.csv'
print(csv_path.stem)                       # '2026'
print(csv_path.read_text(encoding="utf-8"))
root.mkdir(parents=True, exist_ok=True)
for p in root.rglob("*.csv"):
    print("found:", p)

Never build paths with string concatenation or hard-coded separators.

### 14.3 Writing Your Own Context Manager

In [ ]:
import time
from contextlib import contextmanager


@contextmanager
def timed(label):
    start = time.perf_counter()
    try:
        yield
    finally:
        print(f"{label}: {time.perf_counter() - start:.3f}s")


def load_everything():
    time.sleep(0.1)


with timed("load"):
    load_everything()

Or as a class with `__enter__`/`__exit__`. Also useful: `contextlib.suppress`, `contextlib.closing`, `contextlib.ExitStack` for a dynamic number of resources.

---

## 15. Iterators, Generators, and Laziness

### 15.1 Generators

A function containing `yield` returns a generator — values are produced on demand, and memory stays constant.

In [ ]:
from pathlib import Path

Path("huge.txt").write_text("1\n2\n\n3\n", encoding="utf-8")


def parse(line):
    return int(line)


def process(record):
    print("record:", record)


def read_records(path):
    """Yield one parsed record per line without loading the file."""
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield parse(line)


for record in read_records("huge.txt"):
    process(record)

Generator expressions are the inline form and should be preferred over list comprehensions when the result is only iterated once:

In [ ]:
from dataclasses import dataclass


@dataclass
class Row:
    amount: int


rows = [Row(10), Row(20), Row(30)]

total = sum(row.amount for row in rows)          # ✅ no intermediate list
print(total)

total = sum([row.amount for row in rows])        # ⚠️ builds the list first
print(total)

### 15.2 `itertools`

In [ ]:
from itertools import chain, islice, groupby, count, cycle, product, combinations

a, b, c = [1, 2], [3], [4, 5]
stream = range(100)
rows = ["r1", "r2"]
cols = ["c1", "c2"]
records = [("x", 1), ("y", 2), ("x", 3)]


def k(record):
    return record[0]


print(list(chain(a, b, c)))                # concatenate iterables lazily
print(list(islice(stream, 10)))            # first 10 without materializing
print(list(product(rows, cols)))           # cartesian product

# groupby must be sorted by the key first!
for key, group in groupby(sorted(records, key=k), key=k):
    print(key, list(group))

print(list(combinations([1, 2, 3], 2)))
print(list(islice(count(10), 3)), list(islice(cycle("ab"), 5)))

`groupby` on unsorted input is the classic misuse — it groups only *consecutive* equal keys.

### 15.3 Exhaustion

A generator can be consumed exactly once.

In [ ]:
gen = (x for x in range(3))
print(list(gen))      # [0, 1, 2]
print(list(gen))      # [] - already exhausted

If you need the values twice, materialize them (`items = list(gen)`) or make the source re-iterable (a function that returns a fresh generator each call).

---

## 16. Common Pitfalls

A consolidated list. Several are expanded above; these are the ones worth memorizing.

### 16.1 Mutable Defaults

In [ ]:
def f(x, acc=[]): ...        # ❌ shared across calls — use None

### 16.2 Late-Binding Closures

In [ ]:
[lambda: i for i in range(3)]   # ❌ all return 2 — bind with i=i

### 16.3 Aliasing vs. Copying

In [ ]:
a = [1, 2, 3]

b = a           # ❌ same object; mutating b mutates a
b.append(4)
print("aliased:", a)

a = [1, 2, 3]
b = a.copy()    # ✅ shallow copy
b.append(4)
print("copied: ", a, b)

### 16.4 `is` for Value Comparison

```python
if x is 1000:   # ❌ works only for small cached ints
if x == 1000:   # ✅
```

*(Kept as a listing: bare `if` clauses with no body. Modern Python also emits a
`SyntaxWarning` for `is` with a literal, which is itself a useful signal.)*

### 16.5 Float Equality

In [ ]:
0.1 + 0.2 == 0.3            # False
import math
math.isclose(0.1 + 0.2, 0.3)   # ✅ True

from decimal import Decimal
Decimal("0.1") + Decimal("0.2")  # ✅ exact — use for money

### 16.6 Integer Division

In [ ]:
7 / 2     # 3.5  — true division, always float
7 // 2    # 3    — floor division
-7 // 2   # -4   — floors toward negative infinity, not toward zero
-7 % 2    # 1    — sign follows the divisor, unlike C

### 16.7 Modifying a Collection While Iterating It
Covered in §6.5 — build a new collection or iterate over a copy.

### 16.8 Shadowing Builtins and Modules
Naming a file `random.py`, `json.py`, `string.py`, or `email.py` in your project directory breaks the standard-library import of the same name, often with a baffling error.

### 16.9 Chained Assignment of Mutables

In [ ]:
a = b = []      # ❌ both names point to one list
a.append(1)     # b is now [1]

### 16.10 Tuple Requires the Comma

In [ ]:
x = (1)     # int
x = (1,)    # tuple
x = 1,      # also a tuple — an accidental trailing comma silently creates one

### 16.11 String `+=` in a Loop
Quadratic. Use `"".join(parts)` (§8.7).

### 16.12 `except` Ordering

In [ ]:
try: ...
except Exception: ...      # ❌ catches everything first
except ValueError: ...     # unreachable

Order handlers most-specific first.

### 16.13 Boolean Is an Integer

In [ ]:
isinstance(True, int)    # True
sum([True, True, False]) # 2  — occasionally useful, often surprising

### 16.14 Default Encoding
`open(path)` without `encoding=` uses the locale default. Always pass `encoding="utf-8"`.

### 16.15 `range` Is Half-Open
`range(1, 5)` yields 1, 2, 3, 4 — the stop value is excluded. Same for slices.

### 16.16 Deleting from a Dict During Iteration
`RuntimeError: dictionary changed size during iteration`. Iterate `list(d.keys())`.

### 16.17 Class Body Scope
Comprehensions inside a class body cannot see other class attributes:

In [ ]:
class C:
    xs = [1, 2, 3]
    ys = [x * 2 for x in xs]        # ✅ works — the outermost iterable is evaluated eagerly


print(C.ys)

# ❌ The inner loop runs in the comprehension's own scope, which cannot see `xs`
try:
    class D:
        xs = [1, 2, 3]
        zs = [x * n for x in xs for n in xs]
except NameError as exc:
    print(f"NameError: {exc}")

Keep computation out of class bodies.

### 16.18 `sys.path` and Implicit Relative Imports
Python 3 has no implicit relative imports. `import sibling` inside a package fails; use `from . import sibling`.

---

## 17. Performance Notes

Correctness and clarity first — but a few habits cost nothing and matter.

| Habit | Why |
|---|---|
| `set`/`dict` for membership | O(1) vs. O(n) |
| `"".join()` over `+=` | O(n) vs. O(n²) |
| Generators for large streams | constant memory |
| `collections.deque` for queues | O(1) `popleft` vs. O(n) `list.pop(0)` |
| Hoist invariants out of loops | avoids repeated work |
| Local variable lookup | faster than global or attribute lookup in hot loops |
| Vectorize with NumPy/pandas | replaces the interpreter loop with C |
| `functools.cache` on pure functions | memoization in one line |

Measure before optimizing:

In [ ]:
import timeit

print(timeit.timeit("'-'.join(str(n) for n in range(100))", number=10000))

Profile a whole script from the shell rather than in a cell:

```bash
python -m cProfile -s cumtime script.py
```

The usual outcome is that the bottleneck is not where you guessed. Premature optimization that damages readability is a net loss; algorithmic choice (§8.1) almost always dominates micro-optimization.

---

## 18. Testing Conventions

```python
# tests/test_stats.py
import pytest
from myproject.stats import average


def test_average_of_integers():
    assert average([1, 2, 3]) == 2.0


def test_average_rejects_empty_input():
    with pytest.raises(ValueError, match="empty"):
        average([])


@pytest.mark.parametrize(
    "values, expected",
    [
        ([1], 1.0),
        ([1, 2], 1.5),
        ([-1, 1], 0.0),
    ],
)
def test_average_cases(values, expected):
    assert average(values) == pytest.approx(expected)
```

*(Kept as a listing: this is the contents of a test **file**, collected and run
by `pytest` from the command line, not executed cell by cell.)*

Conventions:

- Files named `test_*.py`, functions named `test_*`, in a top-level `tests/` directory.
- One behavior per test; the name states the behavior, so a failure report reads as a sentence.
- Arrange–Act–Assert structure inside each test.
- Use `pytest.approx` for floats, never `==`.
- Test the public interface, not private helpers — otherwise refactoring breaks tests that should not care.
- Use fixtures for shared setup; use `tmp_path` for anything touching the filesystem.

---

## 19. Project Layout and Tooling

### 19.1 Layout

```
myproject/
├── pyproject.toml          # metadata, dependencies, tool configuration
├── README.md
├── .gitignore
├── src/
│   └── myproject/
│       ├── __init__.py
│       ├── __main__.py     # enables `python -m myproject`
│       ├── config.py
│       └── data/
│           ├── __init__.py
│           └── loader.py
└── tests/
    ├── test_config.py
    └── test_loader.py
```

The `src/` layout prevents accidentally importing the package from the working directory instead of the installed copy — a common source of "it works for me" bugs.

### 19.2 Environments

```bash
python3 -m venv .venv
source .venv/bin/activate            # Windows: .venv\Scripts\activate
pip install -e ".[dev]"
pip freeze > requirements.txt
```

One environment per project, never installed globally. `uv` is a faster modern alternative to `pip`/`venv` with the same model.

### 19.3 Tooling

| Tool | Role |
|---|---|
| `black` | Opinionated formatter — ends all formatting debate |
| `ruff` | Very fast linter (and formatter); replaces flake8, isort, pyupgrade |
| `mypy` / `pyright` | Static type checking |
| `pytest` | Testing |
| `pre-commit` | Runs the above on every commit |

Minimal `pyproject.toml`:

```toml
[tool.black]
line-length = 88

[tool.ruff]
line-length = 88
[tool.ruff.lint]
select = ["E", "F", "I", "UP", "B", "SIM"]   # errors, pyflakes, imports,
                                            # pyupgrade, bugbear, simplify

[tool.mypy]
python_version = "3.11"
strict = true

[tool.pytest.ini_options]
testpaths = ["tests"]
```

Adopt a formatter early. Automated formatting removes an entire category of review comment and makes diffs about behavior instead of whitespace.

---

## 20. Quick Reference Checklist

Before committing, check that:

**Structure**
- [ ] Four-space indentation, no tabs
- [ ] Lines within the project's limit; formatter has been run
- [ ] Two blank lines around top-level definitions, one between methods
- [ ] Imports grouped stdlib / third-party / local, no wildcards

**Naming**
- [ ] `snake_case` functions and variables, `CapWords` classes, `UPPER_CASE` constants
- [ ] No shadowed builtins or stdlib module names
- [ ] Names describe meaning; length matches scope

**Correctness**
- [ ] No mutable default arguments
- [ ] No mutation of a collection while iterating it
- [ ] `is` used only for `None`, `True`, `False`, and sentinels
- [ ] Floats compared with `math.isclose`, money with `Decimal`
- [ ] `encoding="utf-8"` on every `open()`
- [ ] Files and resources opened with `with`

**Errors**
- [ ] No bare `except:`, no silent `pass`
- [ ] Exceptions are specific, and messages include the offending value
- [ ] `raise ... from exc` when re-raising with context
- [ ] No `assert` used to validate external input

**Design**
- [ ] Functions do one thing and fit on a screen
- [ ] Guard clauses instead of deep nesting
- [ ] Options passed as keyword-only arguments
- [ ] Data-holding classes are dataclasses; `__repr__` is defined
- [ ] Right data structure chosen (set for membership, dict for lookup)

**Documentation**
- [ ] Public functions, classes, and modules have docstrings
- [ ] Comments explain *why*, and none of them are stale
- [ ] Type hints on public interfaces

**Verification**
- [ ] Tests exist for the new behavior and for the failure cases
- [ ] Linter and type checker pass
- [ ] `if __name__ == "__main__":` guard on anything runnable

---

## Further Reading

- **PEP 8** — Style Guide for Python Code: <https://peps.python.org/pep-0008/>
- **PEP 20** — The Zen of Python: <https://peps.python.org/pep-0020/>
- **PEP 257** — Docstring Conventions: <https://peps.python.org/pep-0257/>
- **PEP 484** — Type Hints: <https://peps.python.org/pep-0484/>
- **Python Language Reference**: <https://docs.python.org/3/reference/>
- **Google Python Style Guide**: <https://google.github.io/styleguide/pyguide.html>

> "Readability counts." — and the reader you are most often writing for is yourself, six months from now, at the moment something has broken.